# Model Interpretability: Understanding What Your Model Learned

Machine learning models are often black boxes. This notebook covers practical techniques to understand, explain, and debug model predictions using SHAP, permutation importance, and partial dependence plots.

**Why this matters:** Regulators, stakeholders, and users demand explanations. Debugging models requires understanding *why* they make certain predictions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)
print('Libraries loaded')


## 1. Train a Model on House Pricing Data


In [ ]:
df = pd.read_csv('data/house_pricing.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
X = pd.get_dummies(df.drop('price', axis=1), columns=['location'], drop_first=True)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'R2 Score: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}')


## 2. Built-in Feature Importance


In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Feature Importance')
plt.title('Random Forest Built-in Feature Importance')
plt.tight_layout()
plt.show()


## 3. Permutation Importance — Model-Agnostic


In [ ]:
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance': perm_result.importances_mean,
    'std': perm_result.importances_std
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(perm_df['feature'], perm_df['importance'], xerr=perm_df['std'])
plt.xlabel('Importance (R2 drop when shuffled)')
plt.title('Permutation Feature Importance')
plt.tight_layout()
plt.show()
perm_df


## 4. Partial Dependence: How Features Affect Predictions


In [ ]:
top_features = importance.tail(3)['feature'].tolist()

fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(rf, X_test, top_features, ax=ax, grid_resolution=30)
plt.suptitle('Partial Dependence Plots')
plt.tight_layout()
plt.show()


## 5. SHAP: Individual Prediction Explanations


In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test[:100])

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test[:100], show=False)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('SHAP not installed. Install with: pip install shap')
    print('\nSHAP shows how each feature pushes predictions up or down')
    print('from the baseline (expected) value for individual instances.')


## 6. Compare Interpretability Across Models


In [ ]:
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gbr.fit(X_train, y_train)

rf_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
gbr_imp = pd.Series(gbr.feature_importances_, index=X.columns).sort_values(ascending=False)

comparison = pd.DataFrame({'RandomForest': rf_imp, 'GradientBoost': gbr_imp})
comparison.head(10).plot(kind='bar', figsize=(12, 5))
plt.title('Feature Importance: Random Forest vs Gradient Boosting')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()


## Key Takeaways


In [ ]:
print('''
- Built-in importance: fast but biased toward high-cardinality features
- Permutation importance: model-agnostic, more reliable
- Partial dependence: shows how predictions change with a feature
- SHAP: explains individual predictions consistently
- Always use multiple interpretability methods and cross-validate
''')
